# Movies ETL Pipeline

This version uses only:
- `movies_metadata.csv`
- `credits.csv`
- TMDB API for US MPA ratings

`keywords.csv` and all keyword-related processing have been removed.


Helper Funtions

In [ ]:
import ast
import pandas as pd


def parse_list(value):
    """Convert a string containing a list of dictionaries into a Python list."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed_value = ast.literal_eval(value)
            if isinstance(parsed_value, list):
                return parsed_value
        except (ValueError, SyntaxError):
            pass

    return []


def extract_names(items):
    """Extract the name value from genres or keywords."""
    names = []
    for item in items:
        if isinstance(item, dict):
            name = item.get("name")
            if name:
                names.append(name)
    return names


def extract_top_cast(cast_list, cast_limit=5):
    """Extract the first cast member names up to cast_limit."""
    cast_names = []
    for person in cast_list[:cast_limit]:
        if isinstance(person, dict):
            name = person.get("name")
            if name:
                cast_names.append(name)
    return cast_names


def extract_director(crew_list):
    """Extract the director's name from the crew list."""
    for person in crew_list:
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name", "")
    return ""


In [ ]:
movies_df = pd.read_csv("movies_metadata.csv",low_memory=False) # Read the movies metadata CSV file into a pandas DataFrame
credits_df = pd.read_csv("credits.csv")


Selecting Columns

In [ ]:
movies_df = movies_df[
    [
        "id",
        "title",
        "runtime",
        "genres"
    ]
].copy()
# Create a copy of the DataFrame with selected columns: 'id', 'title', 'runtime', and 'genres'
credits_df = credits_df[
    [
        "id",
        "cast",
        "crew"
    ]
].copy()

3. Clean movies DataFrame

In [ ]:
movies_df["id"] = pd.to_numeric(movies_df["id"],errors="coerce")

movies_df["runtime"] = pd.to_numeric(movies_df["runtime"], errors="coerce")

movies_df = movies_df.dropna(subset=["id"])

movies_df["id"] = movies_df["id"].astype(int)

movies_df["title"] = (movies_df["title"].fillna("").astype(str).str.strip())

movies_df = movies_df.drop_duplicates(subset=["id"])


# -----------------------------------
# 4. Clean credits DataFrame
# -----------------------------------

credits_df["id"] = pd.to_numeric(credits_df["id"],errors="coerce")

credits_df = credits_df.dropna(subset=["id"])

credits_df["id"] = credits_df["id"].astype(int)

credits_df = credits_df.drop_duplicates(subset=["id"])







Join movies and credits

In [ ]:
final_df = pd.merge(movies_df, credits_df, on="id", how="left")
print("Merge completed.")
print("Rows after merge:", len(final_df))

Check Merge Before Parsing

In [ ]:
print("\nColumns after merge:")

print(final_df.columns.tolist())

print("\nFirst five merged records:")

print(
    final_df[
        [
            "id",
            "title",
            "runtime",
            "genres",
            "cast",
            "crew",
            "keywords"
        ]
    ].head()
)

Transform: Parse Complex Columns

In [ ]:
final_df["genres_parsed"] = final_df["genres"].apply(parse_list)

final_df["cast_parsed"] = final_df["cast"].apply(parse_list)

final_df["crew_parsed"] = final_df["crew"].apply(parse_list)

final_df["keywords_parsed"] = final_df["keywords"].apply(parse_list)

Transform: Extract Simple Values

In [ ]:
final_df["genre_names"] = final_df["genres_parsed"].apply(extract_names)

final_df["top_cast"] = final_df["cast_parsed"].apply(lambda cast_list: extract_top_cast(cast_list,cast_limit=5))

final_df["director"] = final_df["crew_parsed"].apply(extract_director)



Create Clean Final Dataframe

In [ ]:
clean_df = final_df[
    [
        "id",
        "title",
        "runtime",
        "genre_names",
        "top_cast",
        "director"
]
].copy()

TMDB API Code

In [ ]:
import os
import requests

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

if not TMDB_TOKEN:
    raise ValueError(
        "TMDB_TOKEN was not found. Add it to your environment, then restart VS Code."
    )

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}


def get_mpa_rating(tmdb_id):
    """Return the first available US MPA certification from TMDB."""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/release_dates"

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"TMDB request failed for movie {tmdb_id}: {error}")
        return ""

    data = response.json()

    for country in data.get("results", []):
        if country.get("iso_3166_1") == "US":
            for release in country.get("release_dates", []):
                rating = release.get("certification", "").strip()
                if rating:
                    return rating

    return ""


Test it with Toy Story

In [ ]:
toy_story_rating = get_mpa_rating(862)
print("Toy Story MPA rating:", toy_story_rating)

# Test the API on five movies first
test_df = clean_df.head(5).copy()
test_df["mpa_rating"] = test_df["id"].apply(get_mpa_rating)
print(test_df[["id", "title", "mpa_rating"]])

# Add the MPA rating to the dataframe that will be exported
# Note: this makes one TMDB request per movie and may take time for a large dataset.
clean_df["mpa_rating"] = clean_df["id"].apply(get_mpa_rating)

# Put mpa_rating next to runtime in the final output
clean_df = clean_df[
    [
        "id",
        "title",
        "runtime",
        "mpa_rating",
        "genre_names",
        "top_cast",
        "director"
]
]

print("MPA ratings added to clean_df.")
print(clean_df[["title", "mpa_rating"]].head())


Validate Final Data

In [ ]:
print("\nFinal DataFrame columns:")
print(clean_df.columns.tolist())

print("\nFinal row count:", len(clean_df))

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate movie IDs:", clean_df["id"].duplicated().sum())

print("\nMPA rating distribution:")
print(clean_df["mpa_rating"].replace("", "Not available").value_counts(dropna=False))


Display Toy Story

In [ ]:
toy_story_df = clean_df[clean_df["id"] == 862]

print("\nToy Story parsed record:")

print(toy_story_df.to_string(index=False))

LOAD: Export Full DATASET

In [ ]:
from pathlib import Path

output_dir = Path.cwd()
csv_path = output_dir / "final_movies.csv"
json_path = output_dir / "final_movies.json"

clean_df.to_csv(csv_path, index=False)
clean_df.to_json(json_path, orient="records", indent=4, force_ascii=False)

print("Saved full CSV to:", csv_path.resolve())
print("Saved full JSON to:", json_path.resolve())
print("Exported columns:", clean_df.columns.tolist())


Load: Export First 20 Records

In [ ]:
first_20_df = clean_df.head(20).copy()
first_20_path = output_dir / "final_movies_20.json"

first_20_df.to_json(
    first_20_path,
    orient="records",
    indent=4,
    force_ascii=False
)

print("Saved first 20 records to:", first_20_path.resolve())


Completion Message

In [ ]:
print("\nETL pipeline completed successfully.")
print("Output folder:", output_dir.resolve())
